In [38]:
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
import pandas as pd

In [39]:
# df = pd.read_csv(r'C:\\Users\\canel\\OneDrive\Desktop\\ToyData.csv', skiprows=3)
df = pd.read_csv('ToyData.csv', skiprows=3)
print(df.shape)
df.head(20)

(600, 8)


,id,b_no,k_no,reg_type,reg_value,loss,error_x,error_y
0,1,1,1,L2,0.1,0.056293,1.3,0.4
1,2,1,2,L2,0.1,0.194419,0.1,1.1
2,3,1,3,L2,0.1,0.041139,1.4,0.7
3,4,1,4,L2,0.1,0.095682,0.9,0.2
4,5,1,5,L2,0.1,0.173365,0.3,0.4
5,6,1,6,L2,0.1,0.172893,0.2,1.1
6,7,1,7,L2,0.1,-0.005358,1.9,1.5
7,8,1,8,L2,0.1,0.045908,1.1,1.3
8,9,1,9,L2,0.1,-0.002439,1.7,1.7
9,10,1,10,L2,0.1,0.035108,1.1,1.7


In [40]:
unique_b_no = df['b_no'].unique()
unique_reg_types = df['reg_type'].unique()
unique_reg_values = df.groupby('reg_type')['reg_value'].unique()
# unique_reg1, unique_reg2 = unique_reg_values[0], unique_reg_values[1]
print(unique_b_no)
print(unique_reg_types)
print(unique_reg_values)
print(df.columns)

[ 1  2  3  4  5  6  7  8  9 10]
['L2' 'dropout']
reg_type
L2         [0.1, 0.5, 0.9]
dropout    [0.1, 0.5, 0.9]
Name: reg_value, dtype: object
Index(['id', 'b_no', 'k_no', 'reg_type', 'reg_value', 'loss', 'error_x',
       'error_y'],
      dtype='object')


In [41]:
summary_df = pd.DataFrame(columns=['b_no','reg_type', 'reg_value', 'mean_loss', 'stdv_loss'])

for b_value in unique_b_no:
    # print(b_value)

    for reg_type in unique_reg_types:
        # print(reg_type)

        for reg_values in unique_reg_values:
            # print(reg_values)

            for value in reg_values:
                # print(value)
                selected_rows = df[(df['b_no'] == b_value) & (df['reg_type'] == reg_type) & (df['reg_value'] == value)]
                # print(selected_rows.shape)
                # input()
                mean_loss = selected_rows['loss'].mean()
                stdv_loss = selected_rows['loss'].std()
                # print(f'{b_value}, {reg_type}, {reg_value}, {mean_loss}')
                new_row = {'b_no': b_value, 'reg_type': reg_type, 'reg_value': value, 'mean_loss': mean_loss, 'stdv_loss': stdv_loss}
                summary_df = pd.concat([summary_df, pd.DataFrame([new_row])], ignore_index=True)

print(f'Summary shape: {summary_df.shape}')
print(summary_df.head(10))
print(summary_df.tail(10))

Summary shape: (120, 5)
  b_no reg_type  reg_value  mean_loss  stdv_loss
0    1       L2        0.1   0.080701   0.074544
1    1       L2        0.5   0.028852   0.067864
2    1       L2        0.9  -0.061692   0.071866
3    1       L2        0.1   0.080701   0.074544
4    1       L2        0.5   0.028852   0.067864
5    1       L2        0.9  -0.061692   0.071866
6    1  dropout        0.1   0.042837   0.059100
7    1  dropout        0.5  -0.006827   0.091933
8    1  dropout        0.9  -0.059143   0.088354
9    1  dropout        0.1   0.042837   0.059100
    b_no reg_type  reg_value  mean_loss  stdv_loss
110   10       L2        0.9  -0.132648   0.101370
111   10       L2        0.1  -0.022188   0.016553
112   10       L2        0.5  -0.085575   0.079743
113   10       L2        0.9  -0.132648   0.101370
114   10  dropout        0.1  -0.018949   0.030705
115   10  dropout        0.5  -0.095497   0.076924
116   10  dropout        0.9  -0.141085   0.068722
117   10  dropout        0.1 

In [42]:
summary_df.columns

Index(['b_no', 'reg_type', 'reg_value', 'mean_loss', 'stdv_loss'], dtype='object')

In [43]:
from scipy.interpolate import griddata

# Define grid for surface
b_no_range = np.linspace(summary_df['b_no'].min(), summary_df['b_no'].max(), 100)
reg_value_range = np.linspace(summary_df['reg_value'].min(), summary_df['reg_value'].max(), 100)
b_no_grid, reg_value_grid = np.meshgrid(b_no_range, reg_value_range)

# Interpolate mean_loss values for the surface
mean_loss_surface = griddata((summary_df['b_no'], summary_df['reg_value']), 
                              summary_df['mean_loss'], 
                              (b_no_grid, reg_value_grid), 
                              method='cubic')

# Create the 3D surface plot
fig = go.Figure()

# Add scatter data points
fig.add_trace(go.Scatter3d(
    x=summary_df['b_no'],
    y=summary_df['reg_value'],
    z=summary_df['mean_loss'],
    mode='markers',
    marker=dict(
        size=3,
        color='blue',
        opacity=0.5
    ),
    name='Data Points'
))

# Add surface plot
fig.add_trace(go.Surface(
    x=b_no_range,
    y=reg_value_range,
    z=mean_loss_surface,
    colorscale='Jet',
    opacity=0.9
))

# Update layout for better visibility
fig.update_layout(scene=dict(
                    xaxis_title='b_no',
                    yaxis_title='reg_value',
                    zaxis_title='mean_loss'),
                    width=800,  # adjust width
                    height=600, # adjust height
                    title='Mean Loss 3D Surface Plot with Data Points')

# Show the plot
fig.show()


In [44]:
# Define grid for surface
b_no_range = np.linspace(summary_df['b_no'].min(), summary_df['b_no'].max(), 100)
reg_value_range = np.linspace(summary_df['reg_value'].min(), summary_df['reg_value'].max(), 100)
b_no_grid, reg_value_grid = np.meshgrid(b_no_range, reg_value_range)

# Interpolate mean_loss values for the surface
mean_loss_surface = griddata((summary_df['b_no'], summary_df['reg_value']), 
                              summary_df['stdv_loss'], 
                              (b_no_grid, reg_value_grid), 
                              method='cubic')

# Create the 3D surface plot
fig = go.Figure()

# Add scatter data points
fig.add_trace(go.Scatter3d(
    x=summary_df['b_no'],
    y=summary_df['reg_value'],
    z=summary_df['stdv_loss'],
    mode='markers',
    marker=dict(
        size=3,
        color='blue',
        opacity=0.5
    ),
    name='Data Points'
))

# Add surface plot
fig.add_trace(go.Surface(
    x=b_no_range,
    y=reg_value_range,
    z=mean_loss_surface,
    colorscale='Jet',
    opacity=0.9
))

# Update layout for better visibility
fig.update_layout(scene=dict(
                    xaxis_title='b_no',
                    yaxis_title='reg_value',
                    zaxis_title='stdv_loss'),
                    width=800,  # adjust width
                    height=600, # adjust height
                    title='Stdv Loss 3D Surface Plot with Data Points')

# Show the plot
fig.show()


In [45]:
# Group by 'reg_type' and 'reg_value'
grouped = summary_df.groupby('reg_type')#, 'reg_value'])

# Create a dictionary to store DataFrames for each combination
dfs = {}

# Iterate over the groups and create a DataFrame for each combination
for name, group in grouped:
    reg_type = name
    df_name = f"{reg_type}"  # Name for the DataFrame
    dfs[df_name] = group.copy()  # Copy the group DataFrame and store it


In [46]:
for df_name, df in dfs.items():
    print(f"DataFrame: {df_name}")
    print(df.shape)
    print(df)
    print("\n")

DataFrame: L2
(60, 5)
    b_no reg_type  reg_value  mean_loss  stdv_loss
0      1       L2        0.1   0.080701   0.074544
1      1       L2        0.5   0.028852   0.067864
2      1       L2        0.9  -0.061692   0.071866
3      1       L2        0.1   0.080701   0.074544
4      1       L2        0.5   0.028852   0.067864
5      1       L2        0.9  -0.061692   0.071866
12     2       L2        0.1   0.020505   0.054002
13     2       L2        0.5  -0.068629   0.086045
14     2       L2        0.9  -0.121966   0.098760
15     2       L2        0.1   0.020505   0.054002
16     2       L2        0.5  -0.068629   0.086045
17     2       L2        0.9  -0.121966   0.098760
24     3       L2        0.1  -0.018598   0.035146
25     3       L2        0.5  -0.035138   0.075368
26     3       L2        0.9  -0.066350   0.124873
27     3       L2        0.1  -0.018598   0.035146
28     3       L2        0.5  -0.035138   0.075368
29     3       L2        0.9  -0.066350   0.124873
36     4 

In [47]:
# Group by 'reg_type'
grouped_by_reg_type = summary_df.groupby('reg_type')

# Iterate over each group of reg_type
for reg_type, reg_type_group in grouped_by_reg_type:
    # Group by 'reg_value' within each reg_type group
    grouped_by_reg_value = reg_type_group.groupby('reg_value')
    
    # Create a figure for the current reg_type
    fig = go.Figure()
    
    # Iterate over each group of reg_value within the current reg_type group
    for reg_value, reg_value_group in grouped_by_reg_value:
        fig.add_trace(go.Scatter(x=reg_value_group['b_no'], y=reg_value_group['mean_loss'], 
                                 mode='lines+markers', name=reg_value,
                                 marker=dict(size=7)))
    
    fig.update_layout(legend=dict(title="Reg. Value"))
    fig.update_layout(title=f'Iterations vs. mean loss for {reg_type}',
                      xaxis_title='iteration',
                      yaxis_title='mean_loss',
                      width=600,  # Set width of the plot
                      height=400)  # Set height of the plot
    
    # Show the plot for the current reg_type
    fig.show()


In [49]:
# Group by 'reg_type'
grouped_by_reg_type = summary_df.groupby('reg_type')

# Iterate over each group of reg_type
for reg_type, reg_type_group in grouped_by_reg_type:
    # Group by 'reg_value' within each reg_type group
    grouped_by_reg_value = reg_type_group.groupby('reg_value')
    
    # Create a figure for the current reg_type
    fig = go.Figure()
    
    # Iterate over each group of reg_value within the current reg_type group
    for reg_value, reg_value_group in grouped_by_reg_value:
        fig.add_trace(go.Scatter(x=reg_value_group['b_no'], y=reg_value_group['stdv_loss'], 
                                 mode='lines+markers', name=reg_value,
                                 marker=dict(size=7)))
    
    fig.update_layout(legend=dict(title="Reg. Value"))
    fig.update_layout(title=f'Iterations vs. stdv loss for {reg_type}',
                      xaxis_title='iteration',
                      yaxis_title='stdv_loss',
                      width=600,  # Set width of the plot
                      height=400)  # Set height of the plot
    
    # Show the plot for the current reg_type
    fig.show()
